In [7]:
import pandas as pd

# 1. Load the FSD50K ground truth files
# Make sure you downloaded and unzipped FSD50K.ground_truth.zip
dev_df = pd.read_csv("dataset/FSD50K.ground_truth/dev.csv")
eval_df = pd.read_csv("dataset/FSD50K.ground_truth/eval.csv")

# 2. Define the classes relevant to Endangered Species Threat Detection
threat_classes = [
    "Gunshot_and_gunfire",
    "Chainsaw",
    "Explosion",
    "Motor_vehicle_(road)",
    "Engine",
]

# Create a regex pattern to match ANY of the target classes
# This looks for 'Gunshot_and_gunfire' OR 'Chainsaw' OR ...
pattern = "|".join(threat_classes)

# 3. Filter both the development and evaluation datasets
filtered_dev = dev_df[dev_df["labels"].str.contains(pattern, na=False)]
filtered_eval = eval_df[eval_df["labels"].str.contains(pattern, na=False)]

# 4. Get the exact filenames (Freesound IDs + .wav)
dev_fnames = filtered_dev["fname"].astype(str) + ".wav"
eval_fnames = filtered_eval["fname"].astype(str) + ".wav"

print(
    f"Found {len(dev_fnames)} training/dev clips and {len(eval_fnames)} evaluation clips."
)

# 5. Save these lists so you can extract them from the main audio zip files
dev_fnames.to_csv("threat_dev_files.txt", index=False, header=False)
eval_fnames.to_csv("threat_eval_files.txt", index=False, header=False)

Found 1975 training/dev clips and 814 evaluation clips.


/tmp/ipykernel_14445/1424755248.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_dev = dev_df[dev_df["labels"].str.contains(pattern, na=False)]
/tmp/ipykernel_14445/1424755248.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_eval = eval_df[eval_df["labels"].str.contains(pattern, na=False)]


In [ ]:
import shutil
import random
from pathlib import Path


def split_and_copy(list_file, source_dir, dataset_dir, train_ratio=0.8):
    # Setup isolated directories inside your 'dataset' folder
    train_dir = Path(dataset_dir) / "train"
    test_dir = Path(dataset_dir) / "test"
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    # Read all target filenames
    with open(list_file, "r") as file:
        filenames = file.read().splitlines()

    # Shuffle the list with a fixed seed for reproducible splits
    random.seed(42)
    random.shuffle(filenames)

    # Calculate the split index
    split_idx = int(len(filenames) * train_ratio)
    train_files = filenames[:split_idx]
    test_files = filenames[split_idx:]

    def copy_files(files, target_path):
        copied, missing = 0, 0
        for fname in files:
            src_file = Path(source_dir) / fname
            if src_file.exists():
                shutil.copy2(src_file, target_path / fname)
                copied += 1
            else:
                missing += 1
        return copied, missing

    # Execute the file copying
    train_copied, train_miss = copy_files(train_files, train_dir)
    test_copied, test_miss = copy_files(test_files, test_dir)

    print(f"Training set: {train_copied} files copied ({train_miss} missing)")
    print(f"Testing set: {test_copied} files copied ({test_miss} missing)")


# UPDATED PATHS TO MATCH YOUR DIRECTORY
split_and_copy(
    list_file="threat_dev_files.txt",  # Located in the root directory
    source_dir="dataset/FSD50K.dev_audio",  # Located inside 'dataset'
    dataset_dir="split_dataset",  # Outputs to dataset/train and dataset/test
)

Training set: 1580 files copied (0 missing)
Testing set: 395 files copied (0 missing)
